# Forecasting Meteorológico Horario con Deep Learning
## Capítulo 1 — Definición del Problema, Dataset y Marco Metodológico

---

> *"The goal is to turn data into information, and information into insight."*  
> — **Carly Fiorina**

> *"Prediction is very difficult, especially about the future."*  
> — **Niels Bohr**

---

La capacidad de anticipar el comportamiento de la atmósfera horas o días antes es uno de los problemas más antiguos y económicamente relevantes de la ciencia. Este proyecto aborda ese desafío desde la perspectiva del **aprendizaje profundo moderno**, evaluando de forma sistemática y rigurosa siete arquitecturas de redes neuronales sobre datos reales de la red meteorológica de Brasil.

Este notebook establece los **fundamentos del proyecto**: contexto, definición formal, datos y decisiones metodológicas.


## Resumen del Proyecto

<table style="border-collapse:collapse; width:100%; font-size:0.95em;">
<tr style="background:#2c3e50; color:white;">
  <th style="padding:8px 12px; text-align:left;">Dimensión</th>
  <th style="padding:8px 12px; text-align:left;">Detalle</th>
</tr>
<tr style="background:#f8f9fa;">
  <td style="padding:8px 12px;"><strong>Problema</strong></td>
  <td style="padding:8px 12px;">Forecasting de temperatura del aire (°C) a múltiples horizontes temporales</td>
</tr>
<tr>
  <td style="padding:8px 12px;"><strong>Dataset</strong></td>
  <td style="padding:8px 12px;">Red INMET Brasil — 38 estaciones, 24 años (2000–2023), ~2.63M registros horarios</td>
</tr>
<tr style="background:#f8f9fa;">
  <td style="padding:8px 12px;"><strong>Horizontes evaluados</strong></td>
  <td style="padding:8px 12px;">H = 24 h (1 día), 72 h (3 días), 168 h (7 días)</td>
</tr>
<tr>
  <td style="padding:8px 12px;"><strong>Lookback</strong></td>
  <td style="padding:8px 12px;">L = 168 horas (7 días de contexto histórico)</td>
</tr>
<tr style="background:#f8f9fa;">
  <td style="padding:8px 12px;"><strong>Arquitecturas comparadas</strong></td>
  <td style="padding:8px 12px;">7 modelos: Persistence, LSTM, GRU, TCN, N-BEATS, Transformer, TFT</td>
</tr>
<tr>
  <td style="padding:8px 12px;"><strong>Cobertura geográfica</strong></td>
  <td style="padding:8px 12px;">5 regiones climáticas de Brasil (Norte, Nordeste, Centro-Oeste, Sudeste, Sul)</td>
</tr>
<tr style="background:#f8f9fa;">
  <td style="padding:8px 12px;"><strong>Protocolo experimental</strong></td>
  <td style="padding:8px 12px;">2 seeds por modelo · Early stopping · Partición temporal estricta · Test en 2023</td>
</tr>
<tr>
  <td style="padding:8px 12px;"><strong>Métricas principales</strong></td>
  <td style="padding:8px 12px;">RMSE, MAE, R², sMAPE — por horizonte y región climática</td>
</tr>
<tr style="background:#f8f9fa;">
  <td style="padding:8px 12px;"><strong>Infraestructura de cómputo</strong></td>
  <td style="padding:8px 12px;">NVIDIA T4 (Kaggle GPU); ~200 horas de GPU totales</td>
</tr>
</table>


## 1. Motivación y Contexto

### ¿Por qué predecir temperatura horaria?

La **temperatura del aire** es quizás la variable meteorológica de mayor impacto transversal en la sociedad. Sus aplicaciones directas abarcan sectores críticos:

| Sector | Aplicación concreta | Impacto de una mejora del 10% en precisión |
|:---|:---|:---|
| 🌾 **Agricultura** | Planificación de siembra, riego, detección de heladas | Reducción de pérdidas por evento climático extremo |
| ⚡ **Energía** | Predicción de demanda eléctrica, gestión de redes | Menor dependencia de generación de emergencia |
| 🏥 **Salud pública** | Alertas tempranas de olas de calor y frío extremo | Reducción de mortalidad en poblaciones vulnerables |
| ✈️ **Aviación** | Seguridad operacional en aeropuertos y rutas | Reducción de cancelaciones y accidentes |
| 🏗️ **Urbanismo** | Diseño de infraestructura resiliente al clima | Planificación de ciudades más sostenibles |

### Brasil como caso de estudio ideal

Brasil es un caso de estudio **excepcionalmente valioso** por tres razones:

1. **Diversidad climática extrema** — El territorio abarca climas desde el **ecuatorial húmedo** de la Amazonia hasta el **subtropical frío** del sur, pasando por el **semiárido** del Nordeste y el **tropical de altitud** del Cerrado. Un modelo que generalice bien sobre esta heterogeneidad es genuinamente robusto.

2. **Red de monitoreo de alta calidad** — El INMET (Instituto Nacional de Meteorología) opera la red automática más extensa de América del Sur, con datos horarios continuos desde el año 2000.

3. **Relevancia económica** — Brasil es la mayor economía de América Latina y uno de los mayores productores agropecuarios del mundo. Las pérdidas por eventos climáticos extremos ascienden a miles de millones de dólares anuales.

### ¿Por qué Deep Learning y no métodos clásicos?

Los modelos estadísticos clásicos (ARIMA, SARIMA, Prophet, ETS) tienen limitaciones estructurales para este problema:

- **Univariados por diseño** — modelan cada serie de forma independiente, sin explotar las correlaciones entre temperatura, humedad, presión y radiación solar.
- **Linealidad** — incapaces de capturar las complejas no-linealidades de los ciclos diarios y estacionales.
- **Un horizonte a la vez** — requieren un modelo distinto para cada horizonte de predicción.
- **No escalan** — entrenar 38 modelos SARIMA individuales con optimización de hiperparámetros es operacionalmente complejo.

El **Deep Learning** resuelve todas estas limitaciones con una única arquitectura multivariada y multi-horizonte, entrenada end-to-end.


## 2. Arquitecturas Evaluadas

Este proyecto compara siete modelos que representan el **estado del arte en forecasting de series de tiempo**, desde el baseline más simple hasta los modelos transformer especializados:

| # | Modelo | Paradigma | Componente clave | Parámetros aprox. | Referencia |
|:---:|:---|:---|:---|:---:|:---|
| 1 | **Persistence** | Baseline estadístico | $\hat{y}_{t+h} = y_t$ | 0 | — |
| 2 | **LSTM** | Recurrente | Long Short-Term Memory gates | ~500K | Hochreiter & Schmidhuber, 1997 |
| 3 | **GRU** | Recurrente | Gated Recurrent Unit | ~350K | Cho et al., 2014 |
| 4 | **TCN** | Convolucional | Dilated causal convolutions | ~400K | Bai et al., 2018 |
| 5 | **N-BEATS** | Feed-forward | Basis expansion + backcast | ~600K | Oreshkin et al., 2020 |
| 6 | **Transformer** | Atención | Multi-head self-attention | ~800K | Vaswani et al., 2017 |
| 7 | **TFT** | Atención + Recurrente | Gated residual + variable selection | ~1.2M | Lim et al., 2021 |

### Jerarquía de complejidad

```
Persistence (0 params)
    ↓  +complejidad recurrente
LSTM / GRU (~350–500K)
    ↓  +paralelismo convolucional
TCN (~400K)
    ↓  +descomposición de tendencia
N-BEATS (~600K)
    ↓  +mecanismo de atención global
Transformer (~800K)
    ↓  +selección de variables + atención temporal
TFT (~1.2M)                    ← mayor expresividad teórica
```

El objetivo no es solo encontrar el modelo más preciso, sino **cuantificar el trade-off** entre complejidad computacional, tiempo de entrenamiento y ganancia en precisión predictiva.


## 3. Naturaleza del Problema: Series de Tiempo vs. Regresión Estándar

Este trabajo aborda un problema de **forecasting de series de tiempo multivariadas** (*multivariate time series forecasting*). La distinción con un problema de regresión estándar es metodológicamente crítica:

| Característica | Regresión estándar | **Series de tiempo (este trabajo)** |
|:---|:---|:---|
| Supuesto sobre observaciones | Independientes e idénticamente distribuidas (i.i.d.) | **Dependencia temporal estructurada** |
| Orden de los datos | Irrelevante | **Crítico — no se puede alterar** |
| Validación | K-Fold aleatorio | **Partición cronológica estricta** |
| Objetivo | Predecir $y$ dado $\mathbf{x}$ | **Predecir $y_{t+1:t+H}$ dado el historial** |
| Riesgo principal | Sobreajuste por varianza | **Data leakage temporal** |
| Métricas principales | MSE, R² | **RMSE, MAE, sMAPE por horizonte** |

### El problema del Data Leakage Temporal

La **autocorrelación temporal** implica que $y_{t+1}$ está estadísticamente relacionada con $y_t, y_{t-1}, \ldots$ Si se usa K-Fold aleatorio, el modelo observa datos del futuro durante el entrenamiento — aprende a "hacer trampa". Sus métricas en validación son artificialmente optimistas y **no reflejan el rendimiento real en producción**.

Este es el error metodológico más frecuente en proyectos de series de tiempo y puede inflar el rendimiento reportado hasta en un **20–40%**.

### Partición temporal implementada

```
|← ─────────── Train (2000–2021, 22 años) ──────────── →|← Val (2022) →|← Test (2023) →|
                                                          ↑              ↑
                                            Early stopping             Evaluación final
                                            Selección de               (nunca vista)
                                            hiperparámetros
```

> ⚠️ **Garantía anti-leakage:** Los scalers se ajustan **exclusivamente** sobre el conjunto de entrenamiento y se aplican (transform-only) sobre validación y test. Ninguna estadística del futuro contamina el aprendizaje.

### Justificación de la partición fija vs. walk-forward CV

La librería `timeseries-cv` implementa validación cruzada temporal con múltiples folds. En este proyecto se optó por **partición fija** por razones concretas:

1. **Costo computacional:** 7 modelos × 38 estaciones × 2 seeds × 25 epochs ≈ **200+ horas de GPU**. K=5 folds elevaría esto a más de **1.000 horas** — inviable.
2. **Horizonte largo:** Con $H=168$ horas y $L=168$ de lookback, cada fold requiere mínimo 1–2 años de datos, limitando a 3–4 folds significativos.
3. **Conservadurismo estadístico:** Hold-out temporal es el método más conservador — equivale a simular el escenario real de despliegue en producción.


## 4. Definición Formal del Problema

### Notación

Sea una red de $S = 38$ estaciones meteorológicas distribuidas por Brasil. Para cada estación $s \in \{1,\ldots,S\}$ y tiempo $t$ (resolución horaria), se observan:

$$
\mathbf{z}_t^{(s)} = \underbrace{y_t^{(s)}}_{\text{objetivo}} \oplus \underbrace{\mathbf{x}_t^{(s)}}_{\text{covariables } \in \mathbb{R}^6}
$$

donde $y_t^{(s)}$ es la **temperatura del aire a 2 metros** (°C) y $\mathbf{x}_t^{(s)}$ incluye precipitación, humedad, presión, radiación solar, velocidad y dirección del viento.

### Tarea de forecasting

Buscamos aprender un modelo $f_\theta$ tal que:

$$
\hat{\mathbf{y}}_{t+1:t+H}^{(s)} = f_\theta\!\left(\mathbf{z}_{t-L+1:t}^{(s)}\right)
$$

donde:
- $L = 168$ horas — **ventana de lookback** (7 días de contexto histórico)
- $H \in \{24,\, 72,\, 168\}$ horas — **horizontes de predicción** evaluados
- $\theta$ — parámetros aprendidos por descenso de gradiente estocástico

### Función de pérdida (entrenamiento)

$$
\mathcal{L}(\theta) = \frac{1}{N \cdot H} \sum_{i=1}^{N} \sum_{h=1}^{H} \left(\hat{y}_{t_i+h}^{(s)} - y_{t_i+h}^{(s)}\right)^2
$$

### Métricas de evaluación (test)

La evaluación final reporta cuatro métricas complementarias desagregadas por modelo, horizonte $h$ y región climática:

| Métrica | Fórmula | Interpretación |
|:---|:---|:---|
| **RMSE** | $\sqrt{\frac{1}{N}\sum(\hat{y}-y)^2}$ | Error en las mismas unidades que la variable (°C); penaliza outliers |
| **MAE** | $\frac{1}{N}\sum|\hat{y}-y|$ | Error absoluto medio; más robusto a outliers |
| **R²** | $1 - \frac{\sum(\hat{y}-y)^2}{\sum(\bar{y}-y)^2}$ | Fracción de varianza explicada (1 = perfecto, 0 = media, <0 = peor que media) |
| **sMAPE** | $\frac{200}{N}\sum\frac{|\hat{y}-y|}{|\hat{y}|+|y|}$ | Error porcentual simétrico; invariante a escala |

### Partición temporal

| Partición | Período | Años | Uso |
|:---|:---|:---:|:---|
| **Entrenamiento** | Enero 2000 – Diciembre 2021 | 22 | Ajuste de parámetros $\theta$ + scalers |
| **Validación** | Enero 2022 – Diciembre 2022 | 1 | Early stopping; selección de hiperparámetros |
| **Test** | Enero 2023 – Diciembre 2023 | 1 | **Evaluación final** — nunca vista durante entrenamiento |


## 5. Dataset: Red INMET de Brasil

### Instituto Nacional de Meteorología (INMET)

El **Instituto Nacional de Meteorología de Brasil (INMET)** opera la red de estaciones meteorológicas automáticas más extensa de América del Sur. Sus datos son de acceso público y representan el estándar de referencia para estudios climáticos en Brasil.

### Características generales

| Característica | Valor |
|:---|:---|
| **Fuente** | INMET — Red de Estaciones Automáticas (BDMEP) |
| **Estaciones utilizadas** | **38** (seleccionadas por completitud de datos 2000–2023) |
| **Cobertura geográfica** | 5 regiones climáticas de Brasil |
| **Período** | 2000 – 2023 (24 años completos) |
| **Resolución temporal** | Horaria (1 registro/hora/estación) |
| **Total de registros** | **~2.63 millones** de filas |
| **Variables** | 7 (1 objetivo + 6 covariables meteorológicas) |

### Variables meteorológicas

| Variable | Símbolo | Unidad | Rol | Descripción |
|:---|:---:|:---:|:---:|:---|
| `temp_2m` | $y_t$ | °C | ⭐ **Objetivo** | Temperatura del aire a 2 metros de altura |
| `precip` | $x_1$ | mm | Covariable | Precipitación acumulada horaria |
| `humedad` | $x_2$ | % | Covariable | Humedad relativa del aire |
| `presion` | $x_3$ | hPa | Covariable | Presión atmosférica al nivel de la estación |
| `radiacion` | $x_4$ | kJ/m² | Covariable | Radiación solar global |
| `viento_vel` | $x_5$ | m/s | Covariable | Velocidad del viento a 10 metros |
| `viento_dir` | $x_6$ | ° | Covariable | Dirección del viento (0–360°) |

### Distribución geográfica por región climática

| Región | Clima dominante | Köppen | Estaciones | Códigos |
|:---|:---|:---:|:---:|:---|
| 🌿 **Norte** | Ecuatorial húmedo / Amazonia | Af, Am | 7 | A101, A201, A102, A104, A135, A210, A018 |
| 🌵 **Nordeste** | Semiárido / Caatinga | BSh, Aw | 8 | A301, A305, A309, A312, A320, A325, A328, A340 |
| 🌾 **Centro-Oeste** | Cerrado / Tropical estacional | Aw, Cwa | 6 | A001, A002, A719, A756, A901, A923 |
| 🌆 **Sudeste** | Subtropical húmedo / Mata Atlántica | Cfa, Cfb | 10 | A701, A621, A711, A744, A652, A627, A521, A537, A615, A634 |
| ❄️ **Sul** | Subtropical oceánico / Pampa | Cfb, Cfa | 7 | A801, A883, A899, A839, A815, A861, A876 |

Esta distribución geográfica permite evaluar la **generalización climática** de los modelos: ¿es una arquitectura robusta tanto bajo el calor ecuatorial húmedo del Norte (>30°C promedio) como bajo el frío subtropical del Sul (<15°C en invierno)?

### Pipeline de procesamiento de datos

```
Datos crudos INMET (CSV por estación)
        ↓
    [Ingestión]  src/data/ingest_inmet.py
        ↓  Unificación, timestamps, zonas horarias
    [Limpieza]   src/data/clean.py
        ↓  Detección de outliers, interpolación temporal
    [Features]   src/data/features.py
        ↓  Lags temporales, medias móviles, variables cíclicas
    [Partición]  src/data/split.py
        ↓  Partición cronológica estricta (train/val/test)
    [Escalado]   src/data/scalers.py
        ↓  StandardScaler fitteado SOLO en train
    [Windowing]  src/data/windowing.py
        ↓  Ventanas deslizantes (L=168, H=168)
    DataLoader PyTorch  →  Modelo  →  Métricas
```


## 6. Estructura del Proyecto y Roadmap

Este Jupyter Book está organizado como un **pipeline científico reproducible**, donde cada notebook construye sobre el anterior:

| Notebook | Título | Contenido |
|:---:|:---|:---|
| **01** ← *este notebook* | Problema y Dataset | Definición formal, datos INMET, metodología |
| **02** | Exploración de Datos (EDA) | Distribuciones, correlaciones, estacionalidad, visualización geográfica |
| **03** | Preprocesamiento | Limpieza, feature engineering, windowing, escalado |
| **04** | Modelos de Deep Learning | Arquitecturas implementadas: LSTM, GRU, TCN, N-BEATS, Transformer, TFT |
| **05** | Entrenamiento y Validación | Curvas de aprendizaje, early stopping, convergencia por modelo |
| **06** | Benchmark Final | Comparación sistemática, ranking, tests estadísticos, análisis por región |

### Principios de diseño del proyecto

- **Reproducibilidad total** — Semillas fijas, entornos capturados, configuración versionada en YAML
- **Separación estricta de concerns** — `src/` contiene código Python puro; notebooks solo orquestan y visualizan
- **Sin data leakage** — Arquitectura de datos que hace imposible el leakage por construcción
- **Escalabilidad** — El mismo código entrena cualquier modelo sobre cualquier estación cambiando un argumento
- **Evaluación rigurosa** — Tests estadísticos de Diebold-Mariano, Friedman + Nemenyi, Wilcoxon y BDS

---

*Los notebooks siguientes desarrollan cada etapa con detalle. El resultado final —métricas, rankings y análisis estadísticos— se presenta en el **Notebook 06: Benchmark Final**.*


In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name != "Proyecto-final-Deep-learning":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import load_yaml, get_logger

cfg = load_yaml(PROJECT_ROOT / "config" / "config.yaml")
log = get_logger(__name__)
log.info("Proyecto: %s | Device: %s", cfg["project"]["name"], cfg["project"]["device"])
print(f"✅ Entorno configurado | Proyecto: {cfg['project']['name']} | Root: {PROJECT_ROOT}")


In [ ]:
from src.data.ingest_inmet import ingest


In [ ]:
from src.data.clean import clean_station

interim_dir = PROJECT_ROOT / cfg['paths']['data_interim']
station = next(
    (f.stem for f in sorted(interim_dir.glob("*.parquet"))), None
) if interim_dir.exists() else None

if station:
    df_raw = __import__('pandas').read_parquet(interim_dir / f"{station}.parquet")
    print(f"Estación de ejemplo: {station} | Shape: {df_raw.shape}")
    print(f"Período: {df_raw.index.min()} → {df_raw.index.max()}")
    print(f"Variables: {list(df_raw.columns)}")
else:
    print("Datos interim no encontrados — ejecutar pipeline de ingestión primero.")
